# Aula 01 — De DN para reflectância (TOA e BOA) e fusão com a

pancromática

Nessa aula, nosso objetivo é converter a imagem de número digital para
reflectância de superfície (BOA). Como continuação da aula anterior,
vamos partir dos recortes em **números digitais** que a Aula 00 gravou
em `data/raw` e chegar a uma imagem de **2 m em reflectância de
superfície**, pronta para a detecção de nuvens (Aula 02) e para a
substituição de pixels com Sentinel-2 (Aula 03).

O caminho tem quatro etapas:

| etapa | de | para | onde acontece |
|------------------|------------------|------------------|------------------|
| calibração radiométrica | DN | radiância (W m⁻² sr⁻¹ µm⁻¹) | bandas MS de 8 m |
| reflectância TOA | radiância | ρ no topo da atmosfera | bandas MS de 8 m |
| correção atmosférica (6S) | ρ TOA | ρ de superfície (BOA) | bandas MS de 8 m |
| fusão Gram-Schmidt | MS 8 m + PAN 2 m | 4 bandas de 2 m (azul, verde, vermelho, NIR) | grade da pancromática |

A conversão é feita nas bandas multiespectrais **nativas de 8 m, antes
da fusão** (pansharpening). A fusão redistribui valores entre as bandas
e não preserva a linearidade radiométrica de cada banda, então aplicar
os coeficientes de calibração depois dela seria uma aproximação.
Convertendo antes, a fusão passa a operar em reflectância.

Cabe ressaltar que isso **não** torna a fusão radiometricamente
rigorosa: apesar de preservar a média e o desvio das bandas, o
Gram-Schmidt altera um pouco a resposta espectral.

> **Trilha recomendada:** rode localmente, com `pixi run lab`. O
> ambiente do projeto já traz o Py6S e o executável do 6S. No Colab a
> correção BOA também roda, mas exige instalar o conda dentro da sessão.

## Preparando o ambiente

As duas células seguintes só fazem diferença no **Colab**, onde elas
instalam o `condacolab`, que é o que permite trazer o executável do 6S
(um programa em Fortran, que não existe no PyPI). **No pixi local elas
não fazem nada**, porque o ambiente já tem tudo.

> No Colab, a primeira célula **reinicia a sessão**. Isso é normal:
> espere a mensagem de reinício e siga para a célula seguinte. Não rode
> as duas de uma vez.

In [1]:
# --- Colab, parte 1 de 2: instala o conda na sessão (REINICIA o ambiente) ---
import sys

NO_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if NO_COLAB else "local (pixi/Jupyter)")

if NO_COLAB:
    !pip install -q condacolab
    import condacolab
    condacolab.install()   # o ambiente reinicia aqui — é esperado

Ambiente: local (pixi/Jupyter)

In [2]:
# --- Colab, parte 2 de 2: rode DEPOIS do reinício ---
import sys

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:
    # rasterio pelo conda-forge junto com o 6S: misturar o GDAL do pip com o do
    # conda é a origem mais comum de erro difícil de diagnosticar.
    !mamba install -q -y -c conda-forge rasterio pyproj py6s sixs pystac-client requests
    print("pronto")

### Módulo de apoio e arquivos de configuração

Essa célula baixa do repositório o `aulas_cbers.py` e a pasta `config/`
(só quando eles ainda não estão na pasta). O `aulas_cbers.py` guarda, de
forma condensada, as etapas das aulas anteriores, e é o que permite
começar o curso por qualquer aula. A pasta `config/` traz a irradiância
solar (ESUN) do sensor e os parâmetros atmosféricos.

In [3]:
# --- Traz o módulo de apoio e os arquivos de configuração do repositório ---
# aulas_cbers.py guarda, de forma compacta, as etapas das aulas anteriores; é o
# que permite começar o curso por qualquer aula. config/ traz a ESUN do sensor e
# os parâmetros atmosféricos.
import os
from pathlib import Path

REPO = "https://github.com/danfioramonte/geoprocessamento-cbers4a"

if not Path("aulas_cbers.py").exists():
    !rm -rf /tmp/geo_repo
    !git clone -q --depth 1 {REPO} /tmp/geo_repo
    !cp /tmp/geo_repo/colab/aulas_cbers.py .
    !cp -r /tmp/geo_repo/colab/config .

for alvo in ["aulas_cbers.py", "config/esun_wpm.json", "config/atmosfera_rio_claro.json"]:
    print(("ok  " if Path(alvo).exists() else "FALTA"), alvo)

ok   aulas_cbers.py
ok   config/esun_wpm.json
ok   config/atmosfera_rio_claro.json

## 0. Configuração

Aqui definimos os parâmetros da aula. O `CENA_ID` é o identificador da
cena escolhida na Aula 00 (por padrão, a cena de referência do curso).
Se você escolheu outra cena, troque o `CENA_ID` nesta célula; para valer
em todas as aulas de uma vez, altere `CENA_PADRAO` e `AOI_BBOX` no
arquivo `colab/aulas_cbers.py`.

O `AOI_BBOX` **precisa ser o mesmo em todas as aulas**, pois é isso que
permite o corregistro com o AROSICS na Aula 03.

Também ficam aqui a faixa de reflectância aceita no arquivo final
(`PISO_REFLECTANCIA` e `TETO_REFLECTANCIA`, explicadas na seção 4.3) e a
opção `AJUSTAR_AOD`, da seção 4.1.

In [4]:
import aulas_cbers as ac

ac.configurar_gdal()          # ajustes de leitura remota, ANTES do rasterio

import json
import math
from datetime import date, datetime

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import ColorInterp, Resampling
from rasterio.warp import reproject

# ---- parâmetros da aula -----------------------------------------------------
CENA_ID = ac.CENA_PADRAO          # cena de referência do curso (Rio Claro, com nuvem)
AOI_BBOX = ac.AOI_BBOX            # [oeste, sul, leste, norte] em EPSG:4326
# Outra cena ou outra área? No Colab o aulas_cbers.py vem pronto do repositório,
# então a troca é feita aqui mesmo: descomente e ajuste as linhas abaixo. O id
# segue o formato da coluna "id" da Aula 00 (use o mesmo CENA_ID nas Aulas 02 e
# 03), e o "ac.AOI_BBOX" faz o download da cena usar a área nova.
# CENA_ID = "CBERS_4A_WPM_20250927_204_141_L4"
# ac.AOI_BBOX = AOI_BBOX = [-47.6, -22.5, -47.3, -22.2]   # o mesmo bbox da Aula 00
NIVEL = "boa"                     # "boa" (recomendado) ou "toa"
ESCALA = ac.ESCALA                # reflectância 0-1 gravada como int16 x10000
NODATA = ac.NODATA                # -9999

# Faixa física da reflectância GRAVADA. Reflectância negativa não existe na
# natureza: o que aparece abaixo de zero é ruído da correção atmosférica e
# undershoot da fusão sobre alvo escuro. O diagnóstico da seção 4.3 mede esses
# negativos ANTES de gravar — o piso não esconde o sintoma, só evita que o
# produto final carregue valor impossível. Como o NODATA é -9999, pixel ceifado
# continua VÁLIDO (vale 0), não vira buraco.
PISO_REFLECTANCIA = 0.0
TETO_REFLECTANCIA = 1.6

# Busca do AOD pelo critério do alvo escuro (seção 4.1). Custa ~30 s e evita o
# erro mais comum do BOA: corrigir demais e afundar a cena em valores negativos.
AJUSTAR_AOD = False

# No Colab, os dados ficam no seu Google Drive, na mesma pasta em que a Aula 00
# gravou (MyDrive/geoprocessamento-cbers4a/data), e não se perdem quando a
# sessão encerra. Localmente não faz nada. USAR_DRIVE = False usa /content/data.
USAR_DRIVE = True
if USAR_DRIVE:
    ac.usar_drive()
P = ac.pastas()
DATA = ac.data_da_cena(CENA_ID)

print(f"cena  : {CENA_ID}")
print(f"data  : {DATA}")
print(f"nível : {NIVEL.upper()}")
print(f"bbox  : {AOI_BBOX}")
# Caminho RELATIVO de proposito: a saida deste notebook vai para o GitHub,
# e o caminho absoluto entregaria a estrutura de pastas da sua maquina.
# Se precisar do absoluto para depurar, rode P["base"].resolve() a parte.
print(f"dados : {P['base']}")

cena  : CBERS_4A_WPM_20260301_203_141_L4
data  : 20260301
nível : BOA
bbox  : [-47.625969, -22.467835, -47.498227, -22.344048]
dados : data

## 1. As bandas da cena

Se você fez a Aula 00, os arquivos já estão em `data/raw`. Se começou
por aqui, a célula abaixo baixa os mesmos recortes do STAC do INPE
(mesma cena e mesmo bbox). Em seguida, ela lista o tamanho, a resolução,
o tipo de dado e o nodata de cada banda.

In [5]:
arquivos = ac.arquivos_da_cena(CENA_ID)
if len(arquivos) < 5:
    arquivos = ac.preparar_aula_01(CENA_ID)

for asset in ["BAND1", "BAND2", "BAND3", "BAND4", "BAND0"]:
    with rasterio.open(arquivos[asset]) as src:
        print(f"{asset} ({ac.BANDAS[asset]:>5}): {src.width:>5} x {src.height:<5} px "
              f"@ {src.res[0]:g} m | {src.dtypes[0]} | nodata={src.nodata} | {src.crs}")

BAND1 ( blue):  1615 x 1742  px @ 8 m | int16 | nodata=0.0 | EPSG:32723
BAND2 (green):  1615 x 1742  px @ 8 m | int16 | nodata=0.0 | EPSG:32723
BAND3 (  red):  1615 x 1742  px @ 8 m | int16 | nodata=0.0 | EPSG:32723
BAND4 (  nir):  1615 x 1742  px @ 8 m | int16 | nodata=0.0 | EPSG:32723
BAND0 (  pan):  6462 x 6968  px @ 2 m | int16 | nodata=0.0 | EPSG:32723

## 2. O que o XML de metadados traz

Ao lado de cada banda, a Aula 00 baixou o XML do produto. É de lá que
vêm os dados que a conversão precisa:

- **`absoluteCalibrationCoefficient`**: um ganho por banda, que
  transforma DN em radiância;
- **`sunPosition/elevation`**: a elevação do Sol no momento da passagem;
- a **data** da passagem, que define a distância Terra-Sol.

A convenção do coeficiente é **multiplicativa e sem offset**:

$$L = DN \times \text{coeficiente}$$

Pinto et al. (2016) confirmam a forma $L = G \cdot DN + b$ para os
sensores do CBERS e mostram que os interceptos ajustados são
estatisticamente iguais a zero.

A célula abaixo lê o XML e a ESUN do sensor e calcula a distância
Terra-Sol e o cosseno do ângulo zenital solar.

> **ESUN de outro sensor:** os valores lidos na célula abaixo
> (`config/esun_wpm.json`) valem para o CBERS-4A com a câmera WPM. Se
> você adaptar a aula para outro sensor, troque também a ESUN, senão a
> reflectância sai errada sem dar erro nenhum. Para as câmeras do
> CBERS-4A, os valores estão na tabela do [Apêndice 4 da
> CRESDA](https://www.cresda.cn/zgzywxyyzx/zlxz/0107/20250228101153187103775.pdf).

In [6]:
meta = ac.ler_xml(CENA_ID)
esun = ac.ler_esun()          # config/esun_wpm.json (fonte: CRESDA, anexo 4)

quando = date.fromisoformat(meta["data"])
d = ac.distancia_terra_sol(quando)
cos_z = math.cos(math.radians(meta["zenite_solar"]))

print(f"XML             : {Path(meta['xml']).name}")
print(f"data/hora       : {meta['datahora']}")
print(f"órbita/ponto    : {meta['orbita']}/{meta['ponto']}  (nível {meta['nivel_produto']})")
print(f"elevação solar  : {meta['elevacao_solar']:.4f}°")
print(f"zênite solar    : {meta['zenite_solar']:.4f}°  (cos = {cos_z:.5f})")
print(f"distância T-S   : {d:.5f} UA")
print(f"pixel           : {meta['pixel_m']:.1f} m\n")
print(f"{'banda':<10} {'coef. calib.':>14} {'ESUN (W m-2 um-1)':>20}")
for n, nome in ac.BANDAS_NATIVAS.items():
    print(f"{n} {nome:<8} {meta['coeficientes'][n]:>14.6f} {esun[n]:>20.2f}")

XML             : rio_claro_CBERS_4A_WPM_20260301_203_141_L4_BAND0.xml
data/hora       : 2026-03-01T13:03:19.194000
órbita/ponto    : 203/141  (nível 4)
elevação solar  : 54.4088°
zênite solar    : 35.5912°  (cos = 0.81319)
distância T-S   : 0.99045 UA
pixel           : 2.0 m

banda        coef. calib.    ESUN (W m-2 um-1)
1 azul           0.291070              1939.89
2 verde          0.297832              1847.82
3 vermelho       0.232504              1535.15
4 nir            0.178993              1071.41

## 3. DN → radiância → reflectância TOA

$$L_\lambda = DN \times G_\lambda \qquad\qquad
\rho_{TOA} = \frac{\pi \, d^2 \, L_\lambda}{ESUN_\lambda \, \cos\theta_z}$$

Onde $d$ é a distância Terra-Sol em unidades astronômicas e $\theta_z$ é
o ângulo zenital solar (90° menos a elevação). O $ESUN$ é a irradiância
solar exoatmosférica da banda. Os valores do WPM ficam em
`config/esun_wpm.json` e foram publicados pela **CRESDA** (a operadora
chinesa do satélite), a partir da resposta espectral do sensor. Eles
estão disponíveis no [site
oficial](https://www.cresda.cn/zgzywxyyzx/zlxz/0107/20250107110105361912927_pc.html).

Antes da conversão, a célula abaixo trata os pixels sem dado. O produto
usa `0` para “sem dado”. Em TOA nenhuma banda chega a zero sobre a
superfície (o espalhamento atmosférico garante algum sinal até na
sombra), então todo zero é falha, e não alvo escuro. Mas há dois tipos
de falha, e eles pedem tratamentos diferentes:

- **Borda**: zero em **todas** as bandas (fora do recorte da cena), ou
  zero parcial encostado nessa região. Vira nodata.
- **Dropout de detector**: zero em **uma** banda só, num pixel isolado
  no meio da cena, com os vizinhos e a pancromática normais. As outras
  bandas desse pixel estão boas, então ele é **reparado**: o valor da
  banda que falhou é interpolado a partir dos vizinhos válidos da mesma
  banda (`ac.reparar_dropouts`).

Por que não marcar o dropout como nodata? Porque a fusão (seção 7)
amplia cada pixel inválido de 8 m para um buraco de 8 × 8 pixels de 2 m,
ou seja, 16 m de lado: a reamostragem cúbica e a máscara reamostrada
contaminam a vizinhança. Na cena do curso são 43 dropouts, que virariam
43 quadrados sem dado no pansharpening.

Depois, a célula converte DN em radiância e radiância em TOA, e imprime
os percentis de cada banda.

In [7]:
dn, perfil, forma = {}, None, None
for n in sorted(ac.BANDAS_NATIVAS):
    with rasterio.open(arquivos[f"BAND{n}"]) as src:
        if perfil is None:
            perfil, forma = src.profile.copy(), (src.height, src.width)
        elif (src.height, src.width) != forma:
            raise ValueError("as bandas MS não estão na mesma grade")
        dn[n] = src.read(1)

zeros = np.zeros(forma, dtype=bool)
for n in dn:
    zeros |= (dn[n] == 0)

# borda (nodata de verdade) x dropout de detector (reparado pelos vizinhos)
borda, dropout = ac.separar_zeros(dn)
dn = ac.reparar_dropouts(dn, dropout)
valido = ~borda
for n in dn:
    valido &= (dn[n] != 0)

print(f"grade MS : {forma[1]} x {forma[0]} px")
print(f"pixels com zero em alguma banda: {int(zeros.sum()):,}")
print(f"  borda  (viram nodata)        : {int(borda.sum()):,}")
print(f"  dropout (reparados)          : {int(dropout.sum()):,}")
print(f"nodata final: {int((~valido).sum()):,} px ({(~valido).mean():.2%})")

# ---- a conversão propriamente dita ----
radiancia, toa = {}, {}
for n, nome in ac.BANDAS_NATIVAS.items():
    radiancia[n] = ac.dn_para_radiancia(dn[n], meta["coeficientes"][n])
    toa[n] = ac.radiancia_para_toa(radiancia[n], esun[n], d, cos_z)

for n, nome in ac.BANDAS_NATIVAS.items():
    v = toa[n][valido]
    print(f"{nome:<9} TOA: p1={np.percentile(v, 1):.4f}  p50={np.percentile(v, 50):.4f}  "
          f"p99={np.percentile(v, 99):.4f}  máx={v.max():.4f}")

grade MS : 1615 x 1742 px
pixels com zero em alguma banda: 43
  borda  (viram nodata)        : 0
  dropout (reparados)          : 43
nodata final: 0 px (0.00%)
azul      TOA: p1=0.0842  p50=0.1012  p99=0.2775  máx=0.5817
verde     TOA: p1=0.0635  p50=0.0953  p99=0.2724  máx=0.6249
vermelho  TOA: p1=0.0436  p50=0.0740  p99=0.2899  máx=0.5872
nir       TOA: p1=0.0905  p50=0.2596  p99=0.4103  máx=0.6477

### Conferindo antes de seguir

Se a convenção do coeficiente estivesse invertida (divisiva em vez de
multiplicativa), a reflectância passaria de 1,0 na maioria dos pixels, o
que é fisicamente impossível. Se o vermelho e o NIR estivessem trocados,
o NDVI ficaria negativo sobre a vegetação.

Faixas esperadas em TOA:

| alvo            | vermelho          | NIR               | NDVI      |
|-----------------|-------------------|-------------------|-----------|
| vegetação densa | 0,03–0,10         | 0,20–0,45         | 0,45–0,80 |
| solo exposto    | próximos entre si | próximos entre si | 0,05–0,25 |
| água            | —                 | \< 0,06           | negativo  |

A função `sanidade()` abaixo amostra 200 mil pixels válidos, imprime os
percentis de cada banda e do NDVI e avisa quando algo sai da faixa.

In [8]:
def sanidade(refl_por_nome, valido, n_amostra=200_000):
    """Faixas de reflectância e NDVI numa amostra de pixels válidos."""
    idx = np.flatnonzero(valido.ravel())
    if idx.size > n_amostra:
        idx = np.random.default_rng(42).choice(idx, n_amostra, replace=False)
    pct = [1, 5, 50, 95, 99]
    print(f"{'banda':<10} " + " ".join(f"p{p:<7}" for p in pct) + "  frac>1")
    for nome, arr in refl_por_nome.items():
        v = arr.ravel()[idx]
        linha = " ".join(f"{x:<8.4f}" for x in np.percentile(v, pct))
        print(f"{nome:<10} {linha}  {float((v > 1.0).mean()):.4%}")

    r = refl_por_nome["vermelho"].ravel()[idx]
    nir = refl_por_nome["nir"].ravel()[idx]
    den = nir + r
    ndvi = np.where(den > 1e-6, (nir - r) / np.where(den > 1e-6, den, 1), np.nan)
    ndvi = ndvi[np.isfinite(ndvi)]
    print(f"{'NDVI':<10} " + " ".join(f"{x:<8.4f}" for x in np.percentile(ndvi, pct)))

    p95 = float(np.percentile(ndvi, 95))
    if p95 < 0.35:
        print("\n  > NDVI p95 baixo demais para uma cena com vegetação. "
              "Confira a atribuição vermelho/NIR.")
    elif p95 > 0.95:
        print("\n  > NDVI p95 alto demais. Confira os valores de ESUN.")
    else:
        print("\n  > faixas dentro do esperado.")


sanidade({ac.BANDAS_NATIVAS[n]: toa[n] for n in sorted(toa)}, valido)

banda      p1       p5       p50      p95      p99       frac>1
azul       0.0842   0.0870   0.1012   0.1751   0.2809    0.0000%
verde      0.0641   0.0696   0.0953   0.1729   0.2761    0.0000%
vermelho   0.0436   0.0482   0.0740   0.2038   0.2927    0.0000%
nir        0.0912   0.1380   0.2596   0.3603   0.4103    0.0000%
NDVI       -0.0121  0.0693   0.5544   0.7060   0.7405  

  > faixas dentro do esperado.

## 4. TOA → BOA: a correção atmosférica com o 6S

A reflectância TOA ainda carrega o efeito da atmosfera: o
**espalhamento** (que acrescenta brilho, principalmente no azul), a
**absorção** por vapor d’água e ozônio e o efeito de vizinhança. O
**6S** (Vermote et al., 1997) modela esses efeitos e devolve três
coeficientes por banda, usados para fazer a inversão:

$$y = x_a L - x_b \qquad\qquad \rho_{BOA} = \frac{y}{1 + x_c \, y}$$

O $x_b$ é a parcela que a atmosfera **acrescenta** ao sinal (a radiância
de trajetória), e ela é maior no azul. Por isso, depois da correção, o
azul precisa cair; se ele não mudar, o 6S provavelmente não rodou.

Os coeficientes dependem da geometria (Sol e sensor), da altitude do
alvo e do estado da atmosfera no dia: **AOD em 550 nm**, **coluna de
vapor d’água** e **coluna de ozônio**. Esses valores, junto com a
altitude do alvo, ficam em `config/atmosfera_rio_claro.json`.

As condições atmosféricas mudam de um dia para o outro. O código para
obter os valores reais do dia está disponível no Google Earth Engine e é
de autoria própria ([acesse
aqui](https://code.earthengine.google.com/e21fe3b1452d05f15f0ac79594c4ef25)).
Caso você tenha escolhido outra cena, AOI ou data, faça a busca no GEE e
substitua os parâmetros no arquivo antes de continuar.

Em Rio Claro o relevo é suave e uma altitude única resolve. Em áreas com
grande amplitude de relevo, o ideal é dividir a cena em faixas de
altitude com um modelo digital de elevação e rodar o 6S uma vez para
cada faixa.

A célula abaixo confere se o 6S está instalado. Se não estiver, ela
avisa e **segue em TOA** para a aula não travar; nesse caso, todo o
resto do curso vai usar um produto TOA. Se aparecer essa mensagem no
pixi, rode `pixi run check` e confira a linha do 6S.

In [9]:
tem_6s = ac.tem_6s()
print("6S disponível:", tem_6s)

if NIVEL == "boa" and not tem_6s:
    print("\nO Py6S ou o executável 6S não estão instalados.")
    print("  local (pixi): já vem no ambiente — rode 'pixi install' e depois 'pixi run lab'")
    print("  Colab       : rode as duas células de instalação do topo do notebook")
    print("\nSeguindo em TOA para não travar a aula.")
    NIVEL = "toa"

atm = None
if NIVEL == "boa":
    atm = json.loads(Path("config/atmosfera_rio_claro.json").read_text(encoding="utf-8"))
    print(f"\nAOD 550 nm      : {atm['aot550']}")
    print(f"vapor d'água    : {atm['vapor_agua_gcm2']} g/cm²")
    print(f"ozônio          : {atm['ozonio_cmatm']} cm-atm")
    print(f"altitude do alvo: {atm['altitude_alvo_km']} km")
    print(f"perfil aerossol : {atm['perfil_aerossol']}")

6S disponível: True

AOD 550 nm      : 0.0882
vapor d'água    : 2.0007 g/cm²
ozônio          : 0.2533 cm-atm
altitude do alvo: 0.6024 km
perfil aerossol : Continental

### 4.1 Escolhendo o AOD quando você não tem a medida do dia (cenário não ideal)

Dos parâmetros, o **AOD** é o que mais mexe no resultado e o mais
difícil de obter. Quando não temos a medida do dia, dá para estimá-lo a
partir da própria imagem, pelo **critério do alvo escuro**.

Numa cena com água ou sombra densa, o pixel mais escuro tem reflectância
de superfície no azul perto de **zero, mas não negativa** (em torno de
0,005). Se o AOD informado for alto demais, o 6S subtrai mais radiância
de trajetória do que existe e o percentil 1 do azul **fica negativo**.
Se for baixo demais, sobra brilho atmosférico e o percentil 1 fica alto.

Com `AJUSTAR_AOD = True`, a célula abaixo roda o 6S só na banda azul,
com vários valores de AOD, e escolhe o que deixa o alvo escuro mais
perto de 0,005 (leva uns 30 s).

In [10]:
if NIVEL == "boa" and AJUSTAR_AOD:
    ALVO_ESCURO = 0.005          # reflectância esperada do pixel mais escuro no azul
    rng_t = np.random.default_rng(0)
    idx_t = np.flatnonzero(valido.ravel())
    idx_t = rng_t.choice(idx_t, min(300_000, idx_t.size), replace=False)
    L_azul = radiancia[1].ravel()[idx_t]      # banda 1 = azul (numeração nativa)

    testes = {}
    print(f"{'AOD 550':>9} {'p1 azul':>10} {'p50 azul':>10} {'% negativo':>12}")
    for aod in [0.03, 0.05, 0.08, 0.12, 0.16, 0.22]:
        xa, xb, xc = ac.coeficientes_6s(meta, dict(atm, aot550=aod), 1)
        y = xa * L_azul - xb
        r = y / (1.0 + xc * y)
        testes[aod] = (float(np.percentile(r, 1)), float(np.percentile(r, 50)),
                       float((r < 0).mean()))
        print(f"{aod:>9.2f} {testes[aod][0]:>10.4f} {testes[aod][1]:>10.4f} "
              f"{testes[aod][2]:>11.2%}")

    aod_sugerido = min(testes, key=lambda a: abs(testes[a][0] - ALVO_ESCURO))
    print(f"\nno arquivo de configuração: aot550 = {atm['aot550']}")
    print(f"sugestão pelo alvo escuro : aot550 = {aod_sugerido:.2f}  "
          f"(p1 do azul = {testes[aod_sugerido][0]:+.4f})")
    if aod_sugerido in (min(testes), max(testes)):
        print("AVISO: a sugestão caiu na ponta da lista testada. Amplie a lista de AOD "
              "acima — o valor certo pode estar fora dela.")
    atm = dict(atm, aot550=aod_sugerido)
    print(f"aplicado nesta execução   : aot550 = {atm['aot550']:.2f}")
    print("Para congelar a escolha, edite 'aot550' em config/atmosfera_rio_claro.json.")
elif NIVEL == "boa":
    print(f"busca de AOD desligada — usando aot550 = {atm['aot550']} do arquivo.")

busca de AOD desligada — usando aot550 = 0.0882 do arquivo.

### 4.2 A correção, banda a banda

Com os parâmetros definidos, o 6S roda uma vez para cada banda e os
coeficientes $x_a$, $x_b$ e $x_c$ são aplicados sobre a radiância.

In [11]:
refl = {ac.BANDAS_NATIVAS[n]: toa[n].copy() for n in sorted(toa)}
coefs_6s = {}

if NIVEL == "boa":
    print(f"{'banda':<10} {'xa':>12} {'xb':>12} {'xc':>12}")
    for n, nome in ac.BANDAS_NATIVAS.items():
        xa, xb, xc = ac.coeficientes_6s(meta, atm, n)     # uma rodada do 6S por banda
        coefs_6s[nome] = {"xa": xa, "xb": xb, "xc": xc}
        y = xa * radiancia[n] - xb
        refl[nome] = y / (1.0 + xc * y)
        print(f"{nome:<10} {xa:>12.6f} {xb:>12.6f} {xc:>12.6f}")
    print("\nxb é o termo que a atmosfera ACRESCENTA (maior no azul, como esperado).")

banda                xa           xb           xc
azul           0.002430     0.083210     0.138700
verde          0.002480     0.046260     0.094890
vermelho       0.002800     0.023250     0.059710
nir            0.003910     0.010770     0.034280

xb é o termo que a atmosfera ACRESCENTA (maior no azul, como esperado).

### O que a correção mudou

O sinal mais claro de que a correção funcionou: o **azul cai bastante**
(era o mais afetado pelo espalhamento) e o **NDVI sobe** (o vermelho cai
proporcionalmente mais que o NIR). A célula abaixo compara as medianas
de cada banda em TOA e BOA e mostra os histogramas.

In [12]:
if NIVEL == "boa":
    idx = np.flatnonzero(valido.ravel())
    idx = np.random.default_rng(42).choice(idx, min(200_000, idx.size), replace=False)

    print(f"{'banda':<10} {'TOA p50':>10} {'BOA p50':>10} {'variação':>12}")
    for n, nome in ac.BANDAS_NATIVAS.items():
        a = float(np.percentile(toa[n].ravel()[idx], 50))
        b = float(np.percentile(refl[nome].ravel()[idx], 50))
        print(f"{nome:<10} {a:>10.4f} {b:>10.4f} {b - a:>+12.4f}")

    def ndvi_de(r, nir):
        den = nir + r
        return (nir - r) / np.where(den > 1e-6, den, np.nan)

    n_toa = ndvi_de(toa[3].ravel()[idx], toa[4].ravel()[idx])
    n_boa = ndvi_de(refl["vermelho"].ravel()[idx], refl["nir"].ravel()[idx])
    print(f"\nNDVI mediano: TOA {np.nanmedian(n_toa):.4f} -> BOA {np.nanmedian(n_boa):.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for nome, cor in [("azul", "tab:blue"), ("vermelho", "tab:red"), ("nir", "tab:green")]:
        n = [k for k, v in ac.BANDAS_NATIVAS.items() if v == nome][0]
        ax[0].hist(toa[n].ravel()[idx], bins=120, range=(0, 0.6), histtype="step",
                   label=f"{nome} TOA", color=cor, linestyle="--")
        ax[0].hist(refl[nome].ravel()[idx], bins=120, range=(0, 0.6), histtype="step",
                   label=f"{nome} BOA", color=cor)
    ax[0].set_title("Distribuição da reflectância"); ax[0].set_xlabel("ρ"); ax[0].legend(fontsize=8)
    ax[1].hist(n_toa[np.isfinite(n_toa)], bins=120, range=(-0.5, 1), histtype="step",
               label="NDVI TOA", linestyle="--", color="k")
    ax[1].hist(n_boa[np.isfinite(n_boa)], bins=120, range=(-0.5, 1), histtype="step",
               label="NDVI BOA", color="k")
    ax[1].set_title("NDVI"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("Nível TOA — sem comparação a fazer.")

banda         TOA p50    BOA p50     variação
azul           0.1012     0.0424      -0.0588
verde          0.0953     0.0685      -0.0268
vermelho       0.0740     0.0605      -0.0135
nir            0.2596     0.2736      +0.0140

NDVI mediano: TOA 0.5544 -> BOA 0.6414

### 4.3 Reflectância negativa: o que é normal e o que é sintoma

Se você olhar as estatísticas do resultado, vai encontrar alguns
**valores negativos**. Vale separar três situações.

**Ruído.** Reflectância de superfície levemente negativa aparece em
qualquer correção atmosférica: sobre água, sombra e asfalto molhado, o
sinal é tão baixo que o erro do modelo (alguns milésimos) passa do zero.
Os produtos operacionais convivem com isso: o produto de reflectância de
superfície do Landsat Collection 2 (USGS), por exemplo, aceita valores
de **−0,2 a 1,6**.

**Sintoma.** Se a **mediana** de uma banda estiver negativa, ou se
**mais de uns 5%** dos pixels de uma banda ficarem abaixo de zero, não é
ruído: é correção demais, quase sempre por **AOD alto**. Nesse caso,
volte na seção 4.1.

**O que vai para o arquivo.** Como reflectância negativa não existe
fisicamente, o produto é gravado com **piso em 0** (`PISO_REFLECTANCIA`,
seção 0). O diagnóstico abaixo roda **antes** do piso, então ele não
esconde o problema. Como o `NODATA` é −9999, o pixel ajustado continua
sendo **dado válido**, com valor 0.

> **Por que 0 e não −0,2?** O piso em zero introduz um viés positivo
> muito pequeno, e só nos pixels ajustados. Se o seu trabalho precisar
> da cauda negativa (para comparar com produtos Landsat C2, por
> exemplo), use `PISO_REFLECTANCIA = -0.2` e informe isso na
> metodologia.

A tabela abaixo mostra, para cada banda, a porcentagem de negativos, o
percentil 1, a mediana e o mínimo, e indica se o resultado é ruído ou
sintoma.

``` python
def diagnostico_negativos(refl_por_nome, valido):
    print(f"{'banda':<10}{'% < 0':>10}{'p1':>10}{'mediana':>10}{'mínimo':>10}")
    fora = {}
    for nome, arr in refl_por_nome.items():
        v = arr[valido]
        fora[nome] = (float((v < 0).mean()), float(np.percentile(v, 1)),
                      float(np.median(v)), float(v.min()))
        print(f"{nome:<10}{fora[nome][0]:>9.2%}{fora[nome][1]:>10.4f}"
              f"{fora[nome][2]:>10.4f}{fora[nome][3]:>10.4f}")
    return fora


neg = diagnostico_negativos(refl, valido)

pior = max(neg.values(), key=lambda x: x[0])
if any(v[2] < 0 for v in neg.values()):
    print("\n> Alguma banda tem MEDIANA negativa. Isso é correção demais, não ruído:")
    print("  reveja o AOD (seção 4.1) antes de seguir.")
elif pior[0] > 0.05:
    print(f"\n> {pior[0]:.1%} dos pixels de uma banda saíram negativos — acima do que se")
    print("  explica por alvo escuro. O AOD do arquivo provavelmente está alto demais.")
else:
    print("\n> Negativos restritos à cauda escura: comportamento esperado de um produto BOA.")
```

    banda          % < 0        p1   mediana    mínimo
    azul          0.00%    0.0214    0.0424   -0.0068
    verde         0.00%    0.0305    0.0685   -0.0027
    vermelho      0.00%    0.0262    0.0605   -0.0076
    nir           0.00%    0.0890    0.2736    0.0319

    > Negativos restritos à cauda escura: comportamento esperado de um produto BOA.

Tabela 1

## 5. Gravando o composto de 8 m

A saída é gravada em `int16`, com a reflectância multiplicada por
**10.000** (a mesma escala do Sentinel-2 L2A, o que evita conversão na
Aula 03). O nodata passa a ser `-9999`, e não 0, porque agora zero é um
valor de reflectância válido.

As bandas ficam **na mesma ordem em que o WPM captura a cena**: B1 azul,
B2 verde, B3 vermelho e B4 NIR (BGRN), que é também a ordem do
Sentinel-2 (B02, B03, B04, B08). Os nomes das bandas ficam **gravados
dentro do arquivo**, junto com a interpretação de cor (assim o QGIS abre
a imagem em cor natural mesmo com o azul na banda 1). As aulas seguintes
acham cada banda pelo nome, em vez de assumir a posição, o que evita
trocas silenciosas entre bandas.

Junto com o GeoTIFF é gravado um arquivo `.params.json` com tudo o que
foi usado (coeficientes de calibração, ESUN, geometria solar, parâmetros
atmosféricos, coeficientes do 6S e o diagnóstico de negativos), para que
seja possível saber depois como aquele arquivo foi gerado.

In [14]:
ORDEM_SAIDA = ac.ORDEM_SAIDA        # [1, 2, 3, 4] = azul, verde, vermelho, NIR (ordem nativa)
NOMES = ac.NOMES_SAIDA              # ["blue", "green", "red", "nir"]

rgbcomp = P["interim"] / f"rgbcomp_{DATA}_{NIVEL}.tif"
perfil.update(count=4, dtype="int16", nodata=NODATA, compress="deflate", predictor=2,
              tiled=True, blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")

PISO_ESC, TETO_ESC = PISO_REFLECTANCIA * ESCALA, TETO_REFLECTANCIA * ESCALA
cortados = {}

ac.remover_aux(rgbcomp)  # um .aux.xml antigo (QGIS/ArcGIS) não pode se sobrepor aos nomes novos
with rasterio.open(rgbcomp, "w", **perfil) as dst:
    for i, n in enumerate(ORDEM_SAIDA, start=1):
        arr = refl[ac.BANDAS_NATIVAS[n]]
        fora = (arr < PISO_REFLECTANCIA) | (arr > TETO_REFLECTANCIA)
        cortados[NOMES[i - 1]] = int(fora[valido].sum())
        esc = np.clip(np.rint(arr * ESCALA), PISO_ESC, TETO_ESC)
        dst.write(np.where(valido, esc, NODATA).astype("int16"), i)
        dst.set_band_description(i, NOMES[i - 1])
    # qual banda é o vermelho, o verde e o azul (o azul está na banda 1)
    dst.colorinterp = [ColorInterp.blue, ColorInterp.green, ColorInterp.red,
                       ColorInterp.undefined]
    dst.update_tags(ORDEM_BANDAS=ac.ORDEM_BANDAS, NIVEL=NIVEL.upper(), FATOR_ESCALA=str(ESCALA),
                    CENA=CENA_ID, DATA=meta["data"],
                    DROPOUTS_REPARADOS=str(int(dropout.sum())))

# procedência: o que foi usado para gerar este arquivo, em texto, ao lado dele
params = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "aula": "01", "cena": CENA_ID, "data_cena": meta["data"], "nivel": NIVEL,
    "bbox_4326": AOI_BBOX,
    "convencao_coeficiente": "L = DN * absoluteCalibrationCoefficient (offset zero)",
    "coeficientes_calibracao": meta["coeficientes"],
    "esun": esun, "distancia_terra_sol_ua": d, "cos_zenite_solar": cos_z,
    "elevacao_solar": meta["elevacao_solar"],
    "coeficientes_6s": coefs_6s or None, "atmosfera": atm,
    "aod_ajustado_por_alvo_escuro": bool(NIVEL == "boa" and AJUSTAR_AOD),
    "faixa_reflectancia": [PISO_REFLECTANCIA, TETO_REFLECTANCIA],
    "negativos_por_banda": {k: {"frac": v[0], "p1": v[1], "mediana": v[2], "min": v[3]}
                            for k, v in neg.items()},
    "ordem_bandas": ac.ORDEM_BANDAS, "fator_escala": ESCALA, "nodata": NODATA,
    "regra_nodata": "zero em TODAS as bandas MS (borda); dropout isolado reparado pelos vizinhos",
    "px_nodata": int((~valido).sum()), "frac_nodata": float((~valido).mean()),
    "px_dropout_reparados": int(dropout.sum()),
}
rgbcomp.with_suffix(".params.json").write_text(
    json.dumps(params, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Escrito: {rgbcomp}")
print(f"Params : {rgbcomp.with_suffix('.params.json')}")
print(f"Pixels cortados na faixa [{PISO_REFLECTANCIA}, {TETO_REFLECTANCIA}]: {cortados}")

Escrito: data\interim\rgbcomp_20260301_boa.tif
Params : data\interim\rgbcomp_20260301_boa.params.json
Pixels cortados na faixa [0.0, 1.6]: {'blue': 4, 'green': 2, 'red': 2, 'nir': 0}

## 6. Alinhando a multiespectral com a pancromática

A Aula 00 recortou cada banda na sua própria grade: 8 m nas
multiespectrais e 2 m na PAN. As duas cobrem a mesma AOI, mas podem
terminar deslocadas em alguns metros e com uma razão que não é
exatamente 4:1.

O Gram-Schmidt precisa de razão inteira e origens coincidentes. Sem
isso, a fusão gera um deslocamento que se propaga para as etapas
seguintes, inclusive para o corregistro da Aula 03.

A função abaixo resolve os dois problemas: recorta as duas imagens na
extensão comum, alinhada à grade de 8 m, e ajusta a PAN para uma grade
exatamente 4 vezes mais fina.

In [15]:
ms, pan, t_ms, t_pan, crs, nod_ms, FATOR = ac.alinhar_ms_pan(
    rgbcomp, arquivos["BAND0"])

print(f"\nfator MS/PAN     : {FATOR}")
print(f"origem MS  (x, y): {t_ms.c:.1f}, {t_ms.f:.1f}")
print(f"origem PAN (x, y): {t_pan.c:.1f}, {t_pan.f:.1f}   (devem ser iguais)")
print(f"CRS              : {crs}")

MS  alinhado: 1614 x 1742 px @ 8 m
PAN alinhada: 6456 x 6968 px @ 2 m

fator MS/PAN     : 4
origem MS  (x, y): 229780.0, 7526958.0
origem PAN (x, y): 229780.0, 7526958.0   (devem ser iguais)
CRS              : EPSG:32723

## 7. A fusão Gram-Schmidt

O Gram-Schmidt é um método de **substituição de componente** (Laben;
Brower, 2000). A ideia:

1.  simula uma pancromática a partir das bandas multiespectrais, $I$;
2.  ajusta a **média e o desvio** da pancromática real aos dessa
    simulada;
3.  calcula a diferença $\Delta = PAN' - I$, que é o **detalhe
    espacial** que só a PAN tem;
4.  devolve esse detalhe a cada banda com um ganho próprio,
    $MS_i^{fus} = MS_i + g_i \Delta$, onde
    $g_i = \mathrm{cov}(MS_i, I) / \mathrm{var}(I)$.

Como o passo 2 ajusta a escala da PAN, tanto faz ela estar em DN ou em
reflectância. Isso **não** valeria para métodos como Brovey ou IHS, em
que a escala da PAN passa direto para o resultado.

### Como simular a pancromática

A forma clássica do passo 1 usa **pesos fixos**, $I = \sum_i w_i MS_i$,
com R 0,166 \| G 0,167 \| B 0,167 \| NIR 0,5 (os padrões genéricos do
ArcGIS Pro, já que o CBERS-4A não consta na lista de sensores da
ferramenta).

O problema é que esses pesos não representam a resposta espectral do
WPM. Com 0,5 no NIR, $I$ fica dominado pelo infravermelho e **não se
parece com a PAN real**. Com isso, o $\Delta$ passa a carregar
diferenças espectrais, e não só detalhe espacial, e essa diferença,
injetada nas bandas, empurra os alvos escuros (água, sombra) para
valores negativos.

Por isso, usamos a abordagem de Aiazzi, Baronti e Selva (2007): degradar
a PAN para a grade de 8 m e estimar **por mínimos quadrados** os
coeficientes que melhor reproduzem a PAN a partir das próprias bandas
multiespectrais,

$$I = a_0 + \sum_i a_i \, MS_i .$$

Esse método é conhecido como **GSA** (*Gram-Schmidt Adaptive*). Nesta
cena, o desvio de $\Delta$ cai de cerca de 218 (pesos fixos) para 26
(GSA), na escala de 10.000. O que sobra de negativo depois disso é a
cauda escura, tratada com o mesmo piso da seção 4.3.

> Essa implementação em NumPy substitui a ferramenta de pansharpening do
> ArcGIS Pro (`CreatePansharpenedRasterDataset`), que exige licença e só
> roda no Python do próprio ArcGIS. A ferramenta da ESRI grava em
> `uint16` com `NoData = 0`, então lá o pixel negativo vira zero e **se
> torna um buraco** na imagem. Aqui ele vira zero e **continua valendo
> como dado**.

### 7.1 Reamostrando a multiespectral para a grade da PAN

Para a fusão, as bandas multiespectrais precisam estar na mesma grade da
PAN. Elas são reamostradas de 8 m para 2 m por convolução cúbica, e a
máscara de pixels válidos acompanha.

In [16]:
# ---- MS reamostrada para a grade da PAN ----
ms_f = ms.astype("float32")
valido_ms = ~np.all(ms == (nod_ms if nod_ms is not None else NODATA), axis=0)
ms_f[:, ~valido_ms] = np.nan

alt, larg = pan.shape
ms_up = np.empty((4, alt, larg), dtype="float32")
for i in range(4):
    reproject(source=np.nan_to_num(ms_f[i], nan=0.0), destination=ms_up[i],
              src_transform=t_ms, src_crs=crs, dst_transform=t_pan, dst_crs=crs,
              resampling=Resampling.cubic)

val_up = np.empty((alt, larg), dtype="float32")
reproject(source=valido_ms.astype("float32"), destination=val_up,
          src_transform=t_ms, src_crs=crs, dst_transform=t_pan, dst_crs=crs,
          resampling=Resampling.bilinear)
valido_fus = (val_up > 0.999) & (pan > 0)

print(f"grade da fusão: {larg} x {alt} px @ 2 m")
print(f"pixels válidos: {valido_fus.mean() * 100:.2f}%")

grade da fusão: 6456 x 6968 px @ 2 m
pixels válidos: 100.00%

### 7.2 Estimando a pancromática simulada e aplicando a fusão

A célula abaixo define as duas funções da fusão:
`intensidade_por_regressao` (os coeficientes do GSA) e `gram_schmidt` (a
fusão em si). Depois, ela mostra os coeficientes e o R² da regressão,
compara o desvio de $\Delta$ com pesos fixos e com o GSA e aplica a
fusão, imprimindo o ganho de cada banda.

In [17]:
def intensidade_por_regressao(ms_lr, pan, valido_lr, fator):
    """Coeficientes da pancromática simulada, estimados por regressão (GSA).

    ms_lr     : (4, h, w) MS na resolução NATIVA (8 m)
    pan       : (h*fator, w*fator) pancromática na grade fina (2 m)
    valido_lr : (h, w) máscara de pixel válido do MS

    Degrada a PAN para a grade do MS (média de blocos fator x fator) e resolve
    por mínimos quadrados  PAN_degradada ~ a0 + sum(a_i * MS_i).
    """
    n_bandas, h, w = ms_lr.shape
    pan_lp = (pan[:h * fator, :w * fator]
              .reshape(h, fator, w, fator).mean(axis=(1, 3)))
    m = valido_lr & (pan_lp > 0)
    X = np.column_stack([ms_lr[i][m].astype("float64") for i in range(n_bandas)]
                        + [np.ones(int(m.sum()))])
    y = pan_lp[m].astype("float64")
    alpha, *_ = np.linalg.lstsq(X, y, rcond=None)
    r2 = float(1.0 - (y - X @ alpha).var() / max(y.var(), 1e-12))
    return alpha, r2, pan_lp


def gram_schmidt(ms_up, pan, valido, alpha=None, pesos=None):
    """Fusão por substituição de componente (Gram-Schmidt / GSA).

    ms_up : (4, h, w) já na grade da PAN, ordem BGRN (azul, verde, vermelho, NIR)
    pan   : (h, w)
    alpha : coeficientes de `intensidade_por_regressao` (modo GSA).
            Sem eles, cai nos pesos fixos — o Gram-Schmidt clássico.
    """
    if alpha is not None:
        a = np.asarray(alpha, dtype="float32")
        I = np.tensordot(a[:-1], ms_up, axes=(0, 0)) + a[-1]
    else:
        w = np.array([pesos[n] for n in NOMES], dtype="float32")
        w = w / w.sum()
        I = np.tensordot(w, ms_up, axes=(0, 0))

    # estatísticas numa amostra: com dezenas de milhões de pixels, média,
    # desvio e covariância convergem muito antes de usar todos eles
    idx = np.flatnonzero(valido.ravel())
    if idx.size > 2_000_000:
        idx = np.random.default_rng(42).choice(idx, 2_000_000, replace=False)
    pa = pan.ravel()[idx].astype("float64")
    ia = I.ravel()[idx].astype("float64")

    pan_ajustada = (pan - pa.mean()) * (ia.std() / max(pa.std(), 1e-9)) + ia.mean()
    delta = pan_ajustada - I

    ia_c = ia - ia.mean()
    var_I = max(float(ia.var()), 1e-9)
    saida = np.empty_like(ms_up)
    ganhos = {}
    for i, nome in enumerate(NOMES):
        bi = ms_up[i].ravel()[idx].astype("float64")
        g = float(np.dot(bi - bi.mean(), ia_c) / ia_c.size) / var_I
        ganhos[nome] = g
        saida[i] = ms_up[i] + g * delta
    return saida, ganhos, delta


# ---- 1. a intensidade simulada, por regressão (GSA) -------------------------
alpha, r2_pan, _ = intensidade_por_regressao(
    ms.astype("float32"), pan, valido_ms, FATOR)

print("I = a0 + soma(a_i * MS_i), por mínimos quadrados contra a PAN degradada:")
for nome, a in zip(NOMES, alpha[:4]):
    print(f"  a_{nome:<6} {a:+.5f}")
print(f"  a0      {alpha[4]:+.3f}")
print(f"  R² da regressão: {r2_pan:.4f}"
      "   (perto de 1 = a MS explica bem a PAN; longe = descasamento espectral)")

# ---- 2. o quanto isso muda o detalhe injetado ------------------------------
# Comparação barata, só na amostra: qual seria o desvio de delta com os pesos
# fixos do Gram-Schmidt clássico. É esse número que produz os negativos.
_idx = np.flatnonzero(valido_fus.ravel())
if _idx.size > 500_000:
    _idx = np.random.default_rng(7).choice(_idx, 500_000, replace=False)
_pa = pan.ravel()[_idx].astype("float64")
_w = np.array([ac.PESOS_GS[n] for n in NOMES], dtype="float64")
_w = _w / _w.sum()
_I_fixo = sum(_w[i] * ms_up[i].ravel()[_idx].astype("float64") for i in range(4))
_I_reg = (sum(alpha[i] * ms_up[i].ravel()[_idx].astype("float64") for i in range(4))
          + alpha[4])
for rotulo, _I in (("pesos fixos", _I_fixo), ("regressão (GSA)", _I_reg)):
    _paj = (_pa - _pa.mean()) * (_I.std() / max(_pa.std(), 1e-9)) + _I.mean()
    print(f"  desvio de delta com {rotulo:<16}: {(_paj - _I).std():8.1f}")
del _I_fixo, _I_reg, _pa, _idx

# ---- 3. a fusão ------------------------------------------------------------
fus, ganhos, delta_fus = gram_schmidt(ms_up, pan, valido_fus, alpha=alpha)
print("\nganho por banda (cov(MS, I) / var(I)):")
for nome, g in ganhos.items():
    print(f"  {nome:<6} {g:.4f}")

I = a0 + soma(a_i * MS_i), por mínimos quadrados contra a PAN degradada:
  a_blue   +0.00676
  a_green  +0.04796
  a_red    +0.05217
  a_nir    +0.05607
  a0      +49.606
  R² da regressão: 0.9445   (perto de 1 = a MS explica bem a PAN; longe = descasamento espectral)
  desvio de delta com pesos fixos     :    217.1
  desvio de delta com regressão (GSA) :     26.5

ganho por banda (cov(MS, I) / var(I)):
  blue   4.8406
  green  5.4917
  red    6.4754
  nir    6.5281

### 7.3 Gravando o resultado

Mesmo com o GSA, uma pequena parte dos pixels escuros fica abaixo de
zero depois da fusão (é ruído em torno de zero, e não reflectância). A
célula mede esses negativos, grava o arquivo de 2 m com o mesmo piso em
0 e atualiza o `.params.json` com o método, os coeficientes e o número
de pixels ajustados.

In [18]:
pansharp = P["interim"] / f"pansharpening_{DATA}_{NIVEL}.tif"
perfil_fus = dict(driver="GTiff", dtype="int16", count=4, height=alt, width=larg,
                  transform=t_pan, crs=crs, nodata=NODATA, compress="deflate",
                  predictor=2, tiled=True, blockxsize=512, blockysize=512,
                  BIGTIFF="IF_SAFER", interleave="band")

# Mesmo com a intensidade por regressão sobra uma cauda abaixo de zero: o
# detalhe injetado é aditivo, e onde a banda já vale pouco (azul sobre água e
# sombra) a soma atravessa o zero. Isso NÃO é reflectância — é ruído em torno de
# zero. Medimos antes e gravamos com o piso.
print(f"{'banda':<8}{'% < 0':>9}{'mínimo':>10}{'mediana dos negativos':>24}")
neg_fus, ceifados = {}, {}
for i, nome in enumerate(NOMES):
    v = fus[i][valido_fus]
    negativos = v[v < PISO_ESC]
    neg_fus[nome] = float((v < 0).mean())
    ceifados[nome] = int(negativos.size)
    med = float(np.median(negativos)) / ESCALA if negativos.size else 0.0
    print(f"{nome:<8}{neg_fus[nome]:>8.2%}{v.min() / ESCALA:>10.4f}{med:>24.4f}")
print(f"\npiso aplicado em {PISO_REFLECTANCIA:g} — os pixels acima continuam intactos,")
print("e o pixel ceifado vale 0 e segue VÁLIDO (o nodata é -9999, outro valor).")

ac.remover_aux(pansharp)  # um .aux.xml antigo (QGIS/ArcGIS) não pode se sobrepor aos nomes novos
with rasterio.open(pansharp, "w", **perfil_fus) as dst:
    for i in range(4):
        b = np.clip(np.rint(fus[i]), PISO_ESC, TETO_ESC)
        dst.write(np.where(valido_fus, b, NODATA).astype("int16"), i + 1)
        dst.set_band_description(i + 1, NOMES[i])          # carimbo da ordem
    dst.colorinterp = [ColorInterp.blue, ColorInterp.green, ColorInterp.red,
                       ColorInterp.undefined]
    dst.update_tags(ORDEM_BANDAS=ac.ORDEM_BANDAS, NIVEL=NIVEL.upper(), FATOR_ESCALA=str(ESCALA),
                    METODO="Gram-Schmidt (GSA, intensidade por regressao)",
                    CENA=CENA_ID,
                    FAIXA_REFLECTANCIA=f"[{PISO_REFLECTANCIA}, {TETO_REFLECTANCIA}]")

params.update({"etapa": "pansharpening",
               "metodo": "Gram-Schmidt (GSA)",
               "intensidade": "regressao multivariada da PAN degradada sobre as bandas MS",
               "intensidade_coeficientes": dict(zip(NOMES + ["const"],
                                                    [float(a) for a in alpha])),
               "intensidade_r2": float(r2_pan),
               "pesos_gs_classico_nao_usados": ac.PESOS_GS,
               "ganhos": ganhos,
               "negativos_apos_fusao": neg_fus,
               "px_ceifados_no_piso": ceifados,
               "entrada_ms": str(rgbcomp), "entrada_pan": str(arquivos["BAND0"]),
               "saida": str(pansharp)})
pansharp.with_suffix(".params.json").write_text(
    json.dumps(params, indent=2, ensure_ascii=False), encoding="utf-8")

with rasterio.open(pansharp) as src:
    print(f"\nEscrito: {pansharp}")
    print(f"  {src.width} x {src.height} px @ {src.res[0]:g} m | {src.dtypes[0]}")
    print(f"  bandas  : {src.descriptions}")
    print(f"  tamanho : {pansharp.stat().st_size / 1024**2:.1f} MB")

banda       % < 0    mínimo   mediana dos negativos
blue       0.28%   -0.1051                 -0.0037
green      0.02%   -0.0277                 -0.0029
red        0.38%   -0.0548                 -0.0052
nir        0.00%   -0.0069                 -0.0040

piso aplicado em 0 — os pixels acima continuam intactos,
e o pixel ceifado vale 0 e segue VÁLIDO (o nodata é -9999, outro valor).
removido pansharpening_20260301_boa.tif.aux.xml (sobra de uma versão anterior do arquivo)

Escrito: data\interim\pansharpening_20260301_boa.tif
  6456 x 6968 px @ 2 m | int16
  bandas  : ('blue', 'green', 'red', 'nir')
  tamanho : 237.8 MB

## 8. Conferindo o resultado

Duas coisas para olhar: se a fusão **ganhou detalhe** sem alterar as
cores, e se a cena tem **nuvem**, porque é ela que a Aula 02 vai
detectar. A primeira figura mostra a cena inteira em cor natural e falsa
cor; a segunda compara o mesmo trecho antes (8 m) e depois (2 m) da
fusão.

In [19]:
# visão geral, em resolução reduzida
with rasterio.open(pansharp) as src:
    fator = max(1, src.width // 900)
    forma = (3, src.height // fator, src.width // fator)
    # cor natural = vermelho, verde, azul -> bandas (3, 2, 1) na ordem BGRN
    vis = src.read(ac.RGB, out_shape=forma,
                   resampling=Resampling.average).astype("float32")
    vis_nir = src.read(NOMES.index("nir") + 1, out_shape=forma[1:],
                       resampling=Resampling.average).astype("float32")
vis[vis == NODATA] = np.nan

fig, ax = plt.subplots(1, 2, figsize=(15, 7.5))
ax[0].imshow(np.dstack([ac.stretch(vis[0]), ac.stretch(vis[1]), ac.stretch(vis[2])]))
ax[0].set_title(f"Cor natural — {NIVEL.upper()} 2 m")
ax[1].imshow(np.dstack([ac.stretch(vis_nir), ac.stretch(vis[0]), ac.stretch(vis[1])]))
ax[1].set_title("Falsa cor NIR/R/G — vegetação em vermelho, nuvem em branco")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

In [20]:
# zoom: 8 m antes da fusão x 2 m depois, no mesmo pedaço de terreno
lado = 200                                   # em pixels de 8 m -> 1,6 km
r0 = max(0, ms.shape[1] // 2 - lado // 2)
c0 = max(0, ms.shape[2] // 2 - lado // 2)

rgb = [b - 1 for b in ac.RGB]                # vermelho, verde, azul (índices 2, 1, 0)
antes = ms[rgb, r0:r0 + lado, c0:c0 + lado].astype("float32")
antes[antes == NODATA] = np.nan
depois = fus[rgb, r0 * FATOR:(r0 + lado) * FATOR, c0 * FATOR:(c0 + lado) * FATOR]

fig, ax = plt.subplots(1, 2, figsize=(13, 6.6))
ax[0].imshow(np.dstack([ac.stretch(b) for b in antes]))
ax[0].set_title("multiespectral original — 8 m")
ax[1].imshow(np.dstack([ac.stretch(b) for b in depois]))
ax[1].set_title("fusionada com a pancromática — 2 m")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

## O que ficou pronto

    data/interim/rgbcomp_<DATA>_<nivel>.tif          8 m, BGRN, reflectância x10000
    data/interim/pansharpening_<DATA>_<nivel>.tif    2 m, BGRN, reflectância x10000   <- produto da aula
    data/interim/*.params.json                       o que foi usado para gerar cada um

Na **Aula 02**, esse arquivo de 2 m entra no OmniCloudMask, que separa
nuvem espessa, nuvem fina e sombra de nuvem. Na **Aula 03**, os pixels
marcados viram lacunas a serem preenchidas com Sentinel-2
super-resolvido.

## Referências

- AIAZZI, B.; BARONTI, S.; SELVA, M. Improving component substitution
  pansharpening through multivariate regression of MS+Pan data. **IEEE
  Transactions on Geoscience and Remote Sensing**,
  1.  45, n. 10, p. 3230-3239, 2007. DOI: 10.1109/TGRS.2007.901007.
- CRESDA. **Irradiância solar equivalente no topo da atmosfera dos
  sensores civis de observação terrestre** (Anexo 4). Rev. 2024-12-22.
- LABEN, C. A.; BROWER, B. V. **Process for enhancing the spatial
  resolution of multispectral imagery using pan-sharpening**.
  Depositante: Eastman Kodak Company. US 6.011.875. 2000.
- PINTO, C.; PONZONI, F.; CASTRO, R.; LEIGH, L.; MISHRA, N.; AARON, D.;
  HELDER, D. First in-flight radiometric calibration of MUX and WFI
  on-board CBERS-4. **Remote Sensing**, v. 8, n. 5, 405,
  1.  DOI: 10.3390/rs8050405.
- VERMOTE, E. F.; TANRÉ, D.; DEUZÉ, J. L.; HERMAN, M.; MORCRETTE, J.-J.
  Second Simulation of the Satellite Signal in the Solar Spectrum, 6S:
  an overview. **IEEE Transactions on Geoscience and Remote
  Sensing**, v. 35, n. 3, p. 675-686, 1997. DOI: 10.1109/36.581987.
- [Py6S](https://py6s.readthedocs.io) ·
  [rasterio](https://rasterio.readthedocs.io)